# 04 - Ensemble models with default settings
Same data and split as step 03 (train up to 2021, test 2022 onward). Every model is trained on `log1p(next-week cases)`; predictions are converted back with `expm1` and **all scores are on the original case-count scale**.
Each model is a scikit-learn `Pipeline`: the preprocessor (district one-hot + numbers passed through) is fitted on training data only. The exception is CatBoost, which reads `district` natively.
No tuning yet, only sensible defaults, `random_state=42` everywhere. The gap between train and test error is what we watch.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import sys, warnings, numpy as np, pandas as pd
sys.path.append("../src")
from split import *
from evaluate import regression_scores, fit_and_score
from sklearn.pipeline import Pipeline
warnings.filterwarnings("ignore")
df = load_features("../data/processed/features.csv")
train, test = time_split(df)
features = feature_columns(df)
X_train, X_test = train[features], test[features]
y_train_log, y_train_raw, y_test_raw = train[TARGET_LOG], train[TARGET_RAW], test[TARGET_RAW]
rows = []
def run(name, family, model, pipeline=True):
    # wrap the model in the preprocessing Pipeline (not for CatBoost), fit, score, store one table row
    est = Pipeline([("prep", make_preprocessor(features)), ("model", model)]) if pipeline else model
    rows.append(fit_and_score(name, family, est, X_train, y_train_log, X_test, y_train_raw, y_test_raw))
    print(name, "-> test RMSE %.2f | train RMSE %.2f | %.1fs" % (rows[-1]["test_RMSE"], rows[-1]["train_RMSE"], rows[-1]["train_seconds"]))

## (a) Bagging with decision trees
**Bagging** = *bootstrap aggregating*: draw many random samples of the training rows (with repeats), train one tree on each sample, and **average** their answers. One deep tree memorises noise; averaging many different trees cancels much of that noise (it lowers *variance*). Here: 100 plain decision trees, each seeing about 63% of the rows, and all columns.

In [2]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
run("Bagging (decision trees)", "bagging",
    BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42), n_estimators=100, random_state=42, n_jobs=-1))

Bagging (decision trees) -> test RMSE 27.88 | train RMSE 11.75 | 25.3s


## (b) Random Forest
A **Random Forest** is bagging with one extra trick: at every split, each tree may only look at a random subset of the columns. That forces the trees to be different from each other, so averaging them helps even more. It is the assignment's main bagging model. Here: 200 trees, otherwise scikit-learn's defaults (trees grow until the leaves are pure, which is why the training error will be tiny).

In [3]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import RandomForestRegressor
run("Random Forest", "bagging", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))

Random Forest -> test RMSE 27.82 | train RMSE 11.53 | 27.5s


## (c) Gradient Boosting (scikit-learn)
**Boosting** = trees trained **one after another**, each new small tree trying to fix the mistakes (the residuals) the previous trees still make. Unlike bagging, the trees are not independent and each one is deliberately weak and shallow. The final answer is the sum of all trees, each scaled by a small *learning rate*. Boosting mainly lowers *bias* (systematic under- or over-prediction). Defaults: 100 trees of depth 3, learning rate 0.1.

In [4]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import GradientBoostingRegressor
run("Gradient Boosting", "boosting", GradientBoostingRegressor(random_state=42))

Gradient Boosting -> test RMSE 26.05 | train RMSE 27.21 | 25.2s


## (d) XGBoost
**XGBoost** is a fast, carefully engineered version of gradient boosting. Same idea (trees fixing previous errors), plus built-in regularisation that penalises over-complex trees, and clever speed tricks. Here: 300 trees, depth 6, learning rate 0.1.

In [5]:
# AI-GENERATED (Claude via Antigravity)
from xgboost import XGBRegressor
run("XGBoost", "boosting", XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1))

XGBoost -> test RMSE 28.34 | train RMSE 19.02 | 2.1s


## (e) CatBoost (district as a native category)
**CatBoost** is another boosting library. Its speciality is text-like columns: instead of one-hot encoding `district`, it converts each district into a number using only the rows that come *before* it (so a row never sees its own answer). We therefore skip the one-hot step and pass `district` as a categorical column. It also builds balanced, symmetric trees, which are hard to over-fit. Here: 500 trees, depth 6.

In [6]:
# AI-GENERATED (Claude via Antigravity)
from catboost import CatBoostRegressor
run("CatBoost", "boosting",
    CatBoostRegressor(iterations=500, depth=6, learning_rate=0.1, cat_features=["district"], random_state=42, verbose=0, thread_count=-1),
    pipeline=False)

CatBoost -> test RMSE 27.61 | train RMSE 21.96 | 13.1s


## (f) AdaBoost
**AdaBoost** is the oldest boosting method. Each new tree concentrates on the training rows the previous trees got most wrong (those rows get more weight), and the final answer is a weighted vote of all trees. Defaults: 50 shallow trees (depth 3). It is simple but sensitive to outliers, which matters with outbreak weeks.

In [7]:
# AI-GENERATED (Claude via Antigravity)
from sklearn.ensemble import AdaBoostRegressor
run("AdaBoost", "boosting", AdaBoostRegressor(random_state=42))

AdaBoost -> test RMSE 41.29 | train RMSE 61.76 | 8.8s


## Results table (with the two naive baselines)
Baselines need no training. Train RMSE is computed for them too, so the table is complete.

In [8]:
# AI-GENERATED (Claude via Antigravity)
district_avg = train.groupby(DISTRICT)[TARGET_RAW].mean()
naive = {"Baseline: persistence (this week)": ("baseline", train["cases_lag_0"], test["cases_lag_0"]),
         "Baseline: district training average": ("baseline", train[DISTRICT].map(district_avg), test[DISTRICT].map(district_avg))}
for name, (fam, p_tr, p_te) in naive.items():
    tr, te = regression_scores(y_train_raw, p_tr), regression_scores(y_test_raw, p_te)
    rows.append({"model": name, "family": fam, "train_RMSE": tr["RMSE"], "test_RMSE": te["RMSE"],
                 "test_MAE": te["MAE"], "test_R2": te["R2"], "train_seconds": 0.0})
res = pd.DataFrame(rows).sort_values("test_RMSE").reset_index(drop=True)
res["test/train RMSE"] = res["test_RMSE"] / res["train_RMSE"]       # how many times worse on unseen weeks
res.to_csv("../reports/results_default.csv", index=False)
display(res.round(3))

,model,family,train_RMSE,test_RMSE,test_MAE,test_R2,train_seconds,test/train RMSE
0,Gradient Boosting,boosting,27.213,26.054,12.035,0.827,25.182,0.957
1,CatBoost,boosting,21.955,27.613,12.645,0.806,13.074,1.258
2,Baseline: persistence (this week),baseline,28.157,27.677,12.802,0.805,0.000,0.983
3,Random Forest,bagging,11.533,27.821,12.484,0.803,27.480,2.412
4,Bagging (decision trees),bagging,11.746,27.878,12.525,0.802,25.251,2.373
5,XGBoost,boosting,19.025,28.338,12.825,0.796,2.127,1.490
6,AdaBoost,boosting,61.763,41.286,19.563,0.567,8.827,0.668
7,Baseline: district training average,baseline,76.893,48.961,21.906,0.390,0.000,0.637
